In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv("DataSet.csv")

In [ ]:
df.head()

In [ ]:
df.sample(5)

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print(df.isnull().sum())


In [ ]:
print("Total Missing Values:", df.isnull().sum().sum())

In [ ]:
missing = df.isnull().sum()

missing[missing > 0].plot(
    kind='bar',
    figsize=(10,5),
    title='Missing Values Per Column'
)

plt.ylabel('Count')
plt.show()

In [ ]:
df['F3924'].value_counts().plot(
    kind='bar',
    title='Fraud vs Non-Fraud'
)

plt.ylabel('Count')
plt.show()

In [ ]:
target = "F3924"

In [ ]:
bank_features = [
'F115','F321','F527','F531',
'F670','F1692','F2082','F2122',
'F2582','F2678','F2737',
'F2956','F3043','F3836',
'F3887','F3889','F3891',
'F3894'
]

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(df.isnull(),
            cbar=False,
            yticklabels=False)
plt.title("Missing Values")
plt.show()

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

missing_values = df.isnull().sum()

print(missing_values)

In [ ]:
missing_df = pd.DataFrame({
    "Column": df.columns,
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().sum()/len(df))*100
})
missing_df

In [ ]:
missing_percent = (df.isnull().sum()/len(df))*100

high_missing = missing_percent[missing_percent > 50]

print(high_missing.sort_values(ascending=False))

In [ ]:
cont=0
for i in high_missing:
    if i>=50:
        cont+=1
cont

In [ ]:
threshold = 80

cols_to_drop = missing_percent[
    missing_percent > threshold
].index

df = df.drop(columns=cols_to_drop)

print("Dropped:", len(cols_to_drop))
print("Remaining Columns:", df.shape[1])

In [ ]:
df.shape

In [ ]:
bank_features = [
'F115','F321','F527','F531',
'F670','F1692','F2082','F2122',
'F2582','F2678','F2737',
'F2956','F3043','F3836',
'F3887','F3889','F3891',
'F3894'
]

for col in bank_features:
    missing = df[col].isnull().mean() * 100
    print(f"{col}: {missing:.2f}%")

In [ ]:
# Fill numerical columns
num_cols = df.select_dtypes(include=['int64','float64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

In [ ]:
# Fill categorical columns
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
from sklearn.preprocessing import LabelEncoder

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

In [ ]:
from sklearn.feature_selection import mutual_info_classif

X = df.drop(columns=[target])
y = df[target]

mi = mutual_info_classif(X.fillna(0), y)

mi_df = pd.DataFrame({
    'Feature': X.columns,
    'MI_Score': mi
})

mi_df = mi_df.sort_values('MI_Score', ascending=False)

print(mi_df.head(30))

In [ ]:
df.drop('Unnamed: 0', axis=1, inplace=True)

In [ ]:
!pip install xgboost

In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

X = df.drop(target, axis=1)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print(importance_df.head(30))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_imp = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

print(rf_imp.head(30))

In [ ]:
top_mi_features = mi_df.head(30)['Feature'].tolist()


In [ ]:
top_xgb_features = importance_df.head(30)['Feature'].tolist()

In [ ]:
top_rf_features = rf_imp.head(30)['Feature'].tolist()

In [ ]:
final_features = list(set(
    bank_features +
    top_mi_features[:20] +
    top_xgb_features[:20] +
    top_rf_features[:20]
))

In [ ]:
len(final_features)

In [ ]:
print("Bank Features:",
      len(set(final_features) & set(bank_features)))

print("MI Features:",
      len(set(final_features) & set(top_mi_features)))

print("XGB Features:",
      len(set(final_features) & set(top_xgb_features)))

print("RF Features:",
      len(set(final_features) & set(top_rf_features)))

In [ ]:
final_features

In [ ]:
final_features.remove('Unnamed: 0')

In [ ]:
X = df[final_features]
y = df['F3924']

print(X.shape)

In [ ]:
final_df = df[final_features + ['F3924']]

print(final_df.shape)

In [ ]:
dashboard_df = df[final_features].copy()


In [ ]:
plt.figure(figsize=(14,10))

corr = df[final_features[:20]].corr()

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:


outlier_summary = []

for col in final_features:
    
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = df[
        (df[col] < lower) |
        (df[col] > upper)
    ]
    
    outlier_summary.append([
        col,
        len(outliers),
        round(len(outliers)/len(df)*100,2)
    ])

outlier_df = pd.DataFrame(
    outlier_summary,
    columns=[
        'Feature',
        'Outlier_Count',
        'Outlier_Percentage'
    ]
)

outlier_df = outlier_df.sort_values(
    by='Outlier_Percentage',
    ascending=False
)

print(outlier_df)

In [ ]:

import math

n_features = len(final_features)

n_cols = 4
n_rows = math.ceil(n_features / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(20, n_rows*4)
)

axes = axes.flatten()

for i, col in enumerate(final_features):

    sns.boxplot(
        y=df[col],
        ax=axes[i]
    )

    axes[i].set_title(col)

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
suspicious = [
    'F3912',
    'F2230',
    'F3908',
    'F270'
]

X = df[final_features].drop(
    columns=suspicious,
    errors='ignore'
)
y = df['F3924']

X

In [ ]:
final_features = [
    f for f in final_features
    if f not in suspicious
]

print("Total features:", len(final_features))
print(final_features)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    sampling_strategy='auto',
    random_state=42,
    k_neighbors=5
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print(y_train.value_counts())
print(y_train_smote.value_counts())

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:,1]

    print("Accuracy :", accuracy_score(y_test,y_pred))
    print("Precision:", precision_score(y_test,y_pred))
    print("Recall   :", recall_score(y_test,y_pred))
    print("F1 Score :", f1_score(y_test,y_pred))
    print("ROC AUC  :", roc_auc_score(y_test,y_prob))

    print("\nClassification Report")
    print(classification_report(y_test,y_pred))

    return y_pred, y_prob

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=1,
    reg_alpha=0.1,
    reg_lambda=1,
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42
)

xgb.fit(
    X_train_smote,
    y_train_smote
)

y_pred_xgb, y_prob_xgb = evaluate_model(
    xgb,
    X_test,
    y_test
)

In [ ]:
xgb_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb.feature_importances_
})

xgb_importance = xgb_importance.sort_values(
    by='Importance',
    ascending=False
)

print(xgb_importance.head(20))

In [ ]:
pip install catboost

In [ ]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    iterations=500,
    learning_rate=0.03,
    depth=6,
    loss_function='Logloss',
    eval_metric='AUC',
    verbose=False,
    random_state=42
)

cat.fit(
    X_train_smote,
    y_train_smote
)

print("CatBOOST Metrics")
y_pred_cat, y_prob_cat = evaluate_model(
    cat,
    X_test,
    y_test
)

In [ ]:

# Probability of being a mule account
prob = cat.predict_proba(X)[:, 1]

dashboard_df = df[final_features].copy()

dashboard_df['Risk_Probability'] = prob

dashboard_df['Risk_Score'] = (
    dashboard_df['Risk_Probability'] * 100
).round(2)

dashboard_df['Prediction'] = (
    dashboard_df['Risk_Probability'] >= 0.5
).astype(int)

In [ ]:
def risk_level(score):

    if score < 30:
        return "Low Risk"

    elif score < 70:
        return "Medium Risk"

    else:
        return "High Risk"


dashboard_df['Risk_Level'] = (
    dashboard_df['Risk_Score']
    .apply(risk_level)
)

print(
    dashboard_df[
        ['Risk_Score',
         'Risk_Level',
         'Prediction']
    ].head()
)

In [ ]:
dashboard_df.to_csv(
    "dashboard_data.csv",
    index=False
)

In [ ]:
import joblib

joblib.dump(cat, "catboost_model.pkl")
joblib.dump(final_features, "selected_features.pkl")

In [ ]:
print(X.shape)
print(len(X.columns))
print(X.columns.tolist())

In [ ]:
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': cat.feature_importances_
})

importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
)

print(importance_df.head(20))

In [ ]:
importance_df.to_csv(
    "feature_importance.csv",
    index=False
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,8))

sns.barplot(
    data=importance_df.head(20),
    x='Importance',
    y='Feature'
)

plt.title("Top 20 Important Features")
plt.show()

In [ ]:
print(
    dashboard_df['Risk_Level']
    .value_counts()
)

In [ ]:
risk_counts = dashboard_df[
    'Risk_Level'
].value_counts()

plt.figure(figsize=(7,7))

plt.pie(
    risk_counts.values,
    labels=risk_counts.index,
    autopct='%1.1f%%'
)

plt.title("Risk Distribution")
plt.show()

In [ ]:
pip install shap

In [ ]:
import shap
explainer = shap.TreeExplainer(cat)

shap_values = explainer.shap_values(X)

In [ ]:
shap.summary_plot(
    shap_values,
    X
)

In [ ]:
shap.summary_plot(
    shap_values,
    X,
    plot_type='bar'
)

In [ ]:
shap.force_plot(
    explainer.expected_value,
    shap_values[10],
    X.iloc[10],
    matplotlib=True
)

In [ ]:

import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------- INPUT FEATURES ----------------
features = [
'F1381', 'F1597', 'F3919', 'F1057', 'F1863', 'F1171',
'F1692', 'F3807', 'F1393', 'F1495', 'F267', 'F1489',
'F2582', 'F3806', 'F1609', 'F3920', 'F3043', 'F3889',
'F1755', 'F3800', 'F3805', 'F2122', 'F1813', 'F527',
'F1821', 'F115', 'F151', 'F950', 'F531', 'F2082',
'F949', 'F3799', 'F1501', 'F2678', 'F3891', 'F2956',
'F2489', 'F1166', 'F2292', 'F2385', 'F3108', 'F3484',
'F3801', 'F3813', 'F3887', 'F2387', 'F141', 'F2494',
'F1827', 'F670', 'F3532', 'F3894', 'F1705', 'F321',
'F3700', 'F2737', 'F1603', 'F3811', 'F3836'
]

print(f"Enter {len(features)} values separated by spaces:")

values = list(map(float, input().split()))

if len(values) != len(features):
    print(f"Expected {len(features)} values but got {len(values)}")

else:

    # ---------------- CREATE INPUT ----------------
    input_df = pd.DataFrame([values], columns=features)

    # ---------------- PREDICTION ----------------
    prob = cat.predict_proba(input_df)[0][1]

    prediction = int(prob >= 0.5)

    risk_score = round(prob * 100, 2)

    if risk_score < 30:
        risk_level = "Low Risk"
    elif risk_score < 70:
        risk_level = "Medium Risk"
    else:
        risk_level = "High Risk"

    print("\n========== RESULT ==========")
    print("Prediction :", "Mule Account" if prediction == 1 else "Normal Account")
    print("Risk Score :", risk_score)
    print("Risk Level :", risk_level)
    print("Probability:", round(prob, 4))

    # ---------------- SHAP EXPLAINER ----------------
    explainer = shap.TreeExplainer(cat)

    user_shap = explainer.shap_values(input_df)

    # ---------------- TOP FEATURES ----------------
    feature_imp = pd.DataFrame({
        'Feature': input_df.columns,
        'Input_Value': input_df.iloc[0].values,
        'SHAP_Value': user_shap[0]
    })

    feature_imp['Abs_SHAP'] = abs(
        feature_imp['SHAP_Value']
    )

    feature_imp = feature_imp.sort_values(
        by='Abs_SHAP',
        ascending=False
    )

    print("\n===== TOP 10 FEATURES =====")
    print(
        feature_imp[
            ['Feature',
             'Input_Value',
             'SHAP_Value']
        ].head(10)
    )

    # =================================================
    # 1. USER SPECIFIC FEATURE IMPORTANCE
    # =================================================

    plt.figure(figsize=(10,8))

    sns.barplot(
        data=feature_imp.head(10),
        x='SHAP_Value',
        y='Feature'
    )

    plt.title("Top Features Affecting This User")
    plt.show()

    # =================================================
    # 2. USER RISK PIE CHART
    # =================================================

    risk_data = {
        'Low Risk': 0,
        'Medium Risk': 0,
        'High Risk': 0
    }

    risk_data[risk_level] = 1

    plt.figure(figsize=(6,6))

    plt.pie(
        risk_data.values(),
        labels=risk_data.keys(),
        autopct='%1.0f%%'
    )

    plt.title("User Risk Level")
    plt.show()

    # =================================================
    # 3. RISK SCORE BAR
    # =================================================

    plt.figure(figsize=(8,2))

    plt.barh(
        ['Risk Score'],
        [risk_score]
    )

    plt.xlim(0,100)

    plt.title(
        f"Risk Score = {risk_score}"
    )

    plt.show()

    # =================================================
    # 4. SHAP FORCE PLOT
    # =================================================

    shap.force_plot(
        explainer.expected_value,
        user_shap[0],
        input_df.iloc[0],
        matplotlib=True
    )

    # =================================================
    # 5. SHAP WATERFALL PLOT
    # =================================================

    shap.plots.waterfall(
        shap.Explanation(
            values=user_shap[0],
            base_values=explainer.expected_value,
            data=input_df.iloc[0],
            feature_names=input_df.columns
        )
    )

explainer = shap.TreeExplainer(cat)

user_shap = explainer.shap_values(input_df)

feature_imp = pd.DataFrame({
    'Feature': input_df.columns,
    'Input_Value': input_df.iloc[0].values,
    'SHAP_Value': user_shap[0]
})

feature_imp['Abs_SHAP'] = abs(
    feature_imp['SHAP_Value']
)

feature_imp = feature_imp.sort_values(
    by='Abs_SHAP',
    ascending=False
)

print("\nTop Features Affecting This User:")
print(
    feature_imp[
        ['Feature',
         'Input_Value',
         'SHAP_Value']
    ].head(10)
)